# Introduction to SQL with Pokémon

**Learning Objective:** By the end of this notebook, you will know how to use SQL to ask questions of a real database — using the Pokédex as your data source.

**Economics Concept:** Databases are how economists store and query large datasets. SQL (Structured Query Language) is the standard tool for that job — just like Excel is for small datasets. In this notebook you will learn the same basic SQL commands used to query government datasets, census records, and financial databases.

In [ ]:
# Uncomment and run this cell ONCE if you have not installed these packages yet
# !pip install sqlalchemy jupysql

## Setting Up

Before we can query data, we need to load the tools we will use. Think of this as opening the right apps before you start working.

- **pandas** is a Python library for working with tables of data.
- **sqlalchemy** is the library that creates a connection between Python and a SQL database.
- **%load_ext sql** turns on the SQL magic command, which lets us write SQL directly in notebook cells.

In [ ]:
import pandas as pd              # for working with tables of data in Python
import sqlalchemy                # for connecting Python to a SQL database
import matplotlib.pyplot as plt  # for creating charts and visualizations
from pathlib import Path         # for safely building file paths
%load_ext sql

## Connecting to the Pokédex Database

Our data lives in a file called **pokedex.sqlite**. SQLite is a type of SQL database stored as a single file — convenient for learning and for small projects.

The line below creates an **engine**, which is the connection between Python and the database. Think of it like dialing a phone number to reach the database.

In [ ]:
# Create a connection to the pokedex database file
pokedex_engine = sqlalchemy.create_engine('sqlite:///pokedex.sqlite')
pokedex_connection = pokedex_engine.connect()

Now we tell the SQL magic extension to use the connection we just made. After running the next cell, every `%sql` or `%%sql` cell in this notebook will send its query to the Pokédex database.

In [ ]:
# Tell the SQL extension which database to use
%sql pokedex_engine

## Part 1 — Exploring the Database

### What Tables Are in the Pokédex?

A SQL database is organized into **tables**. Each table is like one sheet in a spreadsheet. The Pokédex database has many tables — one for Pokémon species, one for move sets, one for types, and so on.

The query below asks the database to list all of its tables. The special table `sqlite_master` is a built-in SQLite table that stores information about the database itself.

In [ ]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

There are a lot of tables! For this notebook we will focus on the most important ones:

| Table | What it contains |
|-------|------------------|
| `pokemon` | One row per Pokémon form — height, weight, base experience |
| `pokemon_species` | One row per Pokémon species — generation, capture rate, happiness |
| `pokemon_species_names` | The name of each species in many languages |
| `pokemon_types` | Which type(s) each Pokémon belongs to |
| `types` | The list of all Pokémon types (Fire, Water, etc.) |
| `type_names` | The name of each type in many languages |
| `pokemon_stats` | The base stats (HP, Attack, Defense, etc.) for each Pokémon |
| `stats` | The names of each stat |
| `pokemon_evolution` | Which Pokémon evolve into which, and at what level |

### Inspecting a Table's Columns

Before we query a table, it helps to know what columns it has. The `PRAGMA table_info()` command is specific to SQLite. It returns one row for every column in the table, showing the column name and data type.

In [ ]:
%sql PRAGMA table_info(pokemon_species);

The `pokemon_species` table stores one row per Pokémon species. Key columns include:
- **id** — the Pokédex number (1 = Bulbasaur, 4 = Charmander, 7 = Squirtle)
- **identifier** — the species name in lowercase (for example, `bulbasaur`)
- **generation_id** — which game generation introduced this Pokémon
- **capture_rate** — how easy it is to catch (higher = easier, max 255)
- **base_happiness** — how happy the Pokémon is when you first catch it

### Looking at the First Few Rows

The `SELECT *` command means "select every column." The `FROM` clause says which table to pull from. The `LIMIT` clause caps the number of rows returned — this is very useful when exploring a large table.

In [ ]:
%%sql
SELECT *
FROM pokemon_species
LIMIT 10;

The first 10 rows are the original Generation I starters and early-route Pokémon — Bulbasaur through Caterpie. Notice that the `identifier` column uses lowercase names with no spaces.

## Part 2 — SELECT and WHERE: Filtering Rows

### Choosing Specific Columns

Instead of selecting every column with `SELECT *`, we can list only the columns we want. This makes the output easier to read. The query below selects just the Pokédex number, the name, the generation, and the capture rate.

In [ ]:
%%sql
SELECT id, identifier, generation_id, capture_rate
FROM pokemon_species
LIMIT 15;

### Filtering with WHERE

The `WHERE` clause is how we filter rows — like applying a condition. Only rows that satisfy the condition are returned.

In economics, this is equivalent to subsetting your sample: for example, keeping only households below a certain income level, or keeping only firms in a particular industry.

The classic starter Pokémon (Bulbasaur, Charmander, Squirtle, and their equivalents in later generations) all have a capture rate of exactly 45. Let's find them.

In [ ]:
%%sql
SELECT id, identifier, generation_id, capture_rate
FROM pokemon_species
WHERE capture_rate = 45;

Every starter Pokémon is listed above. They all share the same capture rate of 45, which is deliberately low — they are meant to be rare and special.

### Filtering with a Comparison Operator

We can also filter using greater-than (`>`), less-than (`<`), and not-equal (`!=`). Let's find Pokémon that are easy to catch — those with a capture rate above 200.

In [ ]:
%%sql
SELECT id, identifier, generation_id, capture_rate
FROM pokemon_species
WHERE capture_rate > 200
LIMIT 15;

These are the most common Pokémon in the wild. Ratatta, Pidgey, and similar early-game Pokémon are designed to be easy to find and catch, so their capture rate is very high.

### Combining Conditions with AND and OR

We can chain multiple conditions. `AND` means both conditions must be true. `OR` means at least one must be true. Below we find Generation I Pokémon that are also hard to catch (capture rate below 50).

In [ ]:
%%sql
SELECT id, identifier, generation_id, capture_rate
FROM pokemon_species
WHERE generation_id = 1
  AND capture_rate < 50;

These are the rare and powerful Generation I Pokémon — legendary birds, Mewtwo, and the starters. They are hard to catch by design.

## Part 3 — ORDER BY and LIMIT: Sorting Results

### Sorting with ORDER BY

The `ORDER BY` clause sorts the output rows. By default it sorts from smallest to largest (ascending). Adding `DESC` reverses the order to largest first (descending).

Let's rank the original Generation I Pokémon from hardest to easiest to catch.

In [ ]:
%%sql
SELECT id, identifier, capture_rate
FROM pokemon_species
WHERE generation_id = 1
ORDER BY capture_rate ASC
LIMIT 10;

Mewtwo has a capture rate of 3 — the lowest possible. This reflects that it is the rarest and most powerful Pokémon in Generation I.

Now let's flip the order and see which Generation I Pokémon are the easiest to catch.

In [ ]:
%%sql
SELECT id, identifier, capture_rate
FROM pokemon_species
WHERE generation_id = 1
ORDER BY capture_rate DESC
LIMIT 10;

Magikarp, Caterpie, and Weedle top the list with a capture rate of 255 — the maximum possible. These are the weakest early-game Pokémon and are designed to be caught easily.

### Sorting by Multiple Columns

You can sort by more than one column. SQL sorts by the first column first; when there is a tie, it uses the second column to break it. Here we sort by generation (earliest first) and then by capture rate within each generation.

In [ ]:
%%sql
SELECT id, identifier, generation_id, capture_rate
FROM pokemon_species
ORDER BY generation_id ASC, capture_rate ASC
LIMIT 15;

## Part 4 — COUNT and GROUP BY: Aggregating Data

### Counting Rows

The `COUNT(*)` function counts how many rows are returned by a query. This is one of the most common operations in data analysis — answering the question "how many?"

Let's count the total number of Pokémon species in the database.

In [ ]:
%%sql
SELECT COUNT(*) AS total_pokemon_species
FROM pokemon_species;

There are 649 Pokémon species in this database, covering Generations I through V.

### Grouping with GROUP BY

The `GROUP BY` clause is one of the most powerful tools in SQL. It splits the data into groups and then applies an aggregation function (like `COUNT`, `AVG`, or `SUM`) to each group separately.

This is equivalent to a pivot table in Excel or a `groupby` in pandas. In economics, we use this constantly — for example, computing average wages by industry, or counting firms by region.

Let's count how many Pokémon species were introduced in each generation.

In [ ]:
%%sql
SELECT is_baby, COUNT(*) AS number_of_pokemon
FROM pokemon_species
GROUP BY is_baby;

`is_baby` has only two possible values: 0 (not a baby) and 1 (baby Pokémon). GROUP BY split the rows into those two groups and COUNT added up the rows in each. There are only 18 baby Pokémon in the entire Pokédex. Now let's use the same GROUP BY pattern on a column with more groups — generation.

In [ ]:
%%sql
SELECT generation_id, COUNT(*) AS number_of_species
FROM pokemon_species
GROUP BY generation_id
ORDER BY generation_id ASC;

Generation I (Red and Blue) introduced the most Pokémon: 151. Later generations added smaller batches. This tells us something about the game designers' strategy over time — the original games were meant to feel abundant, while later games built on an already large base.

### Average and Max with GROUP BY

We can use other aggregation functions besides `COUNT`. Let's find the average and minimum capture rate per generation. A lower average capture rate means the Pokémon in that generation are harder to catch on average.

In [ ]:
%%sql
SELECT
    generation_id,
    ROUND(AVG(capture_rate), 1) AS average_capture_rate,
    MIN(capture_rate)           AS lowest_capture_rate,
    MAX(capture_rate)           AS highest_capture_rate
FROM pokemon_species
GROUP BY generation_id
ORDER BY generation_id ASC;

Every generation includes at least one nearly-impossible-to-catch Pokémon (capture rate of 3 or close to it) and at least one trivially easy catch (capture rate of 255). The average hovers in a similar range across generations, suggesting consistent game design.

## Part 5 — JOIN: Combining Tables

### Why Do We Need JOINs?

A SQL database stores related information in separate tables. This avoids repeating data and keeps things organized. But to get a complete picture, we often need to **join** two or more tables together.

Think of it like this: one spreadsheet has Pokémon IDs and stats; another spreadsheet has Pokémon IDs and names. To get a table with both names and stats, we need to match the ID columns and merge the two sheets. That is exactly what a JOIN does.

### Getting English Names with JOIN

The `pokemon_species` table stores the species name in the `identifier` column as a lowercase string (e.g. `bulbasaur`). The proper English names with correct capitalization are stored in the `pokemon_species_names` table. We join on the species ID. The language ID 9 means English.

In [ ]:
%%sql
SELECT pokemon_species_id, name
FROM pokemon_species_names
WHERE local_language_id = 9
LIMIT 5;

The `pokemon_species_names` table has a `pokemon_species_id` column that matches the `id` column in `pokemon_species`. That shared column is what we use in the JOIN condition below. Now let's combine the two tables to get the English name alongside the Pokédex number and capture rate.

In [ ]:
%%sql
SELECT
    pokemon_species.id              AS pokedex_number,
    pokemon_species_names.name      AS pokemon_name,
    pokemon_species.generation_id   AS generation,
    pokemon_species.capture_rate    AS capture_rate
FROM pokemon_species
JOIN pokemon_species_names
    ON pokemon_species.id = pokemon_species_names.pokemon_species_id
WHERE pokemon_species_names.local_language_id = 9
LIMIT 15;

Now we can see proper Pokémon names instead of lowercase identifiers. The JOIN matched each species ID in the `pokemon_species` table with the same ID in the `pokemon_species_names` table, and we filtered to keep only the English names.

### Joining Three Tables: Pokemon, Types, and Type Names

Each Pokémon has one or two types (Fire, Water, Grass, etc.). The type information is split across three tables:
- `pokemon_types` — which type ID belongs to which Pokémon ID
- `types` — the type ID and its lowercase identifier  
- `type_names` — the proper English name for each type

We need to join all three to get a readable result. The `slot` column tells us whether it is the first type (slot 1) or the second type (slot 2). Some Pokémon have only one type.

In [ ]:
%%sql
SELECT
    pokemon_species_names.name  AS pokemon_name,
    type_names.name             AS type_name,
    pokemon_types.slot          AS type_slot
FROM pokemon
JOIN pokemon_species_names
    ON pokemon.species_id = pokemon_species_names.pokemon_species_id
    AND pokemon_species_names.local_language_id = 9
JOIN pokemon_types
    ON pokemon.id = pokemon_types.pokemon_id
JOIN type_names
    ON pokemon_types.type_id = type_names.type_id
    AND type_names.local_language_id = 9
WHERE pokemon.is_default = 1
  AND pokemon.id <= 9
ORDER BY pokemon.id, pokemon_types.slot;

The original starter Pokémon have dual types: Bulbasaur is Grass and Poison, while Charmander is pure Fire and Squirtle is pure Water. The `WHERE pokemon.is_default = 1` filter keeps only the default form of each Pokémon, removing alternate forms like Mega Evolutions.

## Part 6 — GROUP BY with JOIN: Answering Bigger Questions

### Which Type Has the Most Pokémon?

Now we combine everything we have learned: JOINs to connect the tables and GROUP BY to count. This query answers a real analytical question: which Pokémon type is most represented in the Pokédex?

We filter to `slot = 1` to count each Pokémon's primary type only, so dual-type Pokémon are not double-counted.

In [ ]:
%%sql
SELECT type_id, COUNT(*) AS number_of_pokemon
FROM pokemon_types
WHERE slot = 1
GROUP BY type_id
ORDER BY number_of_pokemon DESC
LIMIT 5;

We get type IDs, not type names. That is fine as a first step — it tells us which numeric IDs are the most common primary types. Type IDs are hard to read, though. In the next query we JOIN to the `type_names` table so we can see the actual name of each type.

In [ ]:
%%sql
SELECT
    type_names.name          AS primary_type,
    COUNT(*)                 AS number_of_pokemon
FROM pokemon
JOIN pokemon_types
    ON pokemon.id = pokemon_types.pokemon_id
    AND pokemon_types.slot = 1
JOIN type_names
    ON pokemon_types.type_id = type_names.type_id
    AND type_names.local_language_id = 9
WHERE pokemon.is_default = 1
GROUP BY type_names.name
ORDER BY number_of_pokemon DESC;

Water is the most common primary type, followed by Normal and Grass. This is not a coincidence — Water-type Pokémon appear throughout the games as common wild Pokémon near water routes, making them an accessible type for beginning players.

### Which Type Has the Highest Average Attack Stat?

Now let's ask a more strategic question: which primary type tends to have the most powerful attack stat on average? This requires joining the stats table as well.

The `pokemon_stats` table has a `stat_id` column. Stat ID 2 = Attack. We join this with the type tables to compute the average attack per type.

In [ ]:
%%sql
SELECT
    type_names.name                      AS primary_type,
    ROUND(AVG(pokemon_stats.base_stat), 1) AS average_attack_stat,
    COUNT(*)                              AS number_of_pokemon
FROM pokemon
JOIN pokemon_types
    ON pokemon.id = pokemon_types.pokemon_id
    AND pokemon_types.slot = 1
JOIN type_names
    ON pokemon_types.type_id = type_names.type_id
    AND type_names.local_language_id = 9
JOIN pokemon_stats
    ON pokemon.id = pokemon_stats.pokemon_id
    AND pokemon_stats.stat_id = 2
WHERE pokemon.is_default = 1
GROUP BY type_names.name
ORDER BY average_attack_stat DESC;

Dragon and Fighting types have the highest average attack stat, which matches the game lore — these types are known for physical power. Normal-type Pokémon have a lower average attack, consistent with them being common, all-purpose Pokémon rather than specialized fighters.

### Who Are the Highest-HP Pokémon?

HP (Hit Points) determines how much damage a Pokémon can take before fainting. Stat ID 1 = HP. Let's find the top 10 tankiest Pokémon.

In [ ]:
%%sql
SELECT
    pokemon_species_names.name  AS pokemon_name,
    pokemon_stats.base_stat     AS hp_stat
FROM pokemon
JOIN pokemon_species_names
    ON pokemon.species_id = pokemon_species_names.pokemon_species_id
    AND pokemon_species_names.local_language_id = 9
JOIN pokemon_stats
    ON pokemon.id = pokemon_stats.pokemon_id
    AND pokemon_stats.stat_id = 1
WHERE pokemon.is_default = 1
ORDER BY hp_stat DESC
LIMIT 10;

Blissey leads with 255 HP — by far the highest in the game. Chansey is Blissey's pre-evolution and also makes the list. These are the go-to defensive Pokémon in competitive play because they can absorb enormous amounts of damage.

## Part 7 — Evolution Chains

### Which Pokémon Evolve and at What Level?

The `pokemon_evolution` table records how Pokémon evolve. The `evolution_trigger_id = 1` filter keeps only level-up evolutions (as opposed to trading or using an item). We join the species names table twice — once for the pre-evolved Pokémon and once for the evolved form.

In [ ]:
%%sql
SELECT
    pre_evolution_names.name   AS evolves_from,
    evolved_names.name         AS evolves_into,
    pokemon_evolution.minimum_level AS level_required
FROM pokemon_evolution
JOIN pokemon_species AS evolved_species
    ON pokemon_evolution.evolved_species_id = evolved_species.id
JOIN pokemon_species_names AS evolved_names
    ON evolved_species.id = evolved_names.pokemon_species_id
    AND evolved_names.local_language_id = 9
JOIN pokemon_species AS pre_evolution_species
    ON evolved_species.evolves_from_species_id = pre_evolution_species.id
JOIN pokemon_species_names AS pre_evolution_names
    ON pre_evolution_species.id = pre_evolution_names.pokemon_species_id
    AND pre_evolution_names.local_language_id = 9
WHERE pokemon_evolution.evolution_trigger_id = 1
  AND pokemon_evolution.minimum_level IS NOT NULL
ORDER BY pokemon_evolution.minimum_level ASC
LIMIT 15;

Caterpie evolves into Metapod at level 7 — one of the earliest evolutions in the game. Notice that the starters (Bulbasaur → Ivysaur at level 16, Charmander → Charmeleon at level 16) appear near the top as well.

## Part 8 — Saving Results to a Pandas DataFrame

### Capturing SQL Results in Python

The `<<` operator saves the result of a SQL query into a Python variable. We can then call `.DataFrame()` on that variable to convert the result into a pandas DataFrame. This is useful when you want to do further analysis or plotting in Python after the SQL query retrieves the data.

In [ ]:
%%sql pokemon_type_counts <<
SELECT
    type_names.name   AS primary_type,
    COUNT(*)          AS number_of_pokemon
FROM pokemon
JOIN pokemon_types
    ON pokemon.id = pokemon_types.pokemon_id
    AND pokemon_types.slot = 1
JOIN type_names
    ON pokemon_types.type_id = type_names.type_id
    AND type_names.local_language_id = 9
WHERE pokemon.is_default = 1
GROUP BY type_names.name
ORDER BY number_of_pokemon DESC;

Now we convert the saved SQL result into a pandas DataFrame. Once it is a DataFrame, we can use any Python data tools — plotting, statistics, further filtering — on the results.

In [ ]:
# Convert the SQL result to a pandas DataFrame
pokemon_type_counts_table = pokemon_type_counts.DataFrame()
pokemon_type_counts_table.head(10)

### Plotting the Results

Now that the data is in a pandas DataFrame, we can create a simple bar chart to visualize the distribution of Pokémon across types. This is a common workflow in data analysis: use SQL to extract and summarize data, then use Python to visualize it.

In [ ]:

# Sort so the largest bars appear first
pokemon_type_counts_sorted = pokemon_type_counts_table.sort_values('number_of_pokemon', ascending=True)

# Build the horizontal bar chart
fig, ax = plt.subplots(figsize=(9, 7))

ax.barh(
    pokemon_type_counts_sorted['primary_type'],
    pokemon_type_counts_sorted['number_of_pokemon'],
    color='steelblue'
)

ax.set_xlabel('Number of Pokémon')
ax.set_title('Number of Pokémon by Primary Type')
plt.tight_layout()
plt.show()

The chart confirms what we found in SQL: Water and Normal dominate the Pokédex, while Ice and Flying types are much rarer. This distribution is intentional — game designers need enough common Pokémon to fill routes and caves, so Normal and Water appear everywhere.

## Part 9 — Practice Queries

Try these queries on your own to practice what you have learned. Each one builds on a concept from the notebook.

**Challenge 1:** Find all Pokémon from Generation II with a `base_happiness` greater than 100. Show their names (from `pokemon_species_names`), generation, and happiness.

In [ ]:
%%sql
SELECT
    pokemon_species_names.name    AS pokemon_name,
    pokemon_species.generation_id AS generation,
    pokemon_species.base_happiness AS base_happiness
FROM pokemon_species
JOIN pokemon_species_names
    ON pokemon_species.id = pokemon_species_names.pokemon_species_id
    AND pokemon_species_names.local_language_id = 9
WHERE pokemon_species.generation_id = 2
  AND pokemon_species.base_happiness > 100
ORDER BY pokemon_species.base_happiness DESC;

**Challenge 2:** Count the number of Pokémon in each generation that are baby Pokémon (`is_baby = 1`). Order by generation.

In [ ]:
%%sql
SELECT
    generation_id,
    COUNT(*) AS number_of_baby_pokemon
FROM pokemon_species
WHERE is_baby = 1
GROUP BY generation_id
ORDER BY generation_id ASC;

**Challenge 3:** Find the 10 fastest Pokémon (stat ID 6 = Speed). Show the Pokémon name and speed stat.

In [ ]:
%%sql
SELECT
    pokemon_species_names.name  AS pokemon_name,
    pokemon_stats.base_stat     AS speed_stat
FROM pokemon
JOIN pokemon_species_names
    ON pokemon.species_id = pokemon_species_names.pokemon_species_id
    AND pokemon_species_names.local_language_id = 9
JOIN pokemon_stats
    ON pokemon.id = pokemon_stats.pokemon_id
    AND pokemon_stats.stat_id = 6
WHERE pokemon.is_default = 1
ORDER BY speed_stat DESC
LIMIT 10;

## Summary

In this notebook you learned the core building blocks of SQL by exploring the Pokédex database:

| SQL Keyword | What it does | Pokémon example |
|-------------|-------------|------------------|
| `SELECT` | Choose which columns to return | Pick just `name` and `capture_rate` |
| `FROM` | Choose which table to query | Query `pokemon_species` |
| `WHERE` | Filter rows by a condition | Only starter Pokémon (`capture_rate = 45`) |
| `ORDER BY` | Sort the results | Rank by HP from highest to lowest |
| `LIMIT` | Cap the number of rows | Show only the top 10 |
| `COUNT` | Count the matching rows | How many Pokémon per generation? |
| `AVG`, `MIN`, `MAX` | Summarize a numeric column | Average attack stat by type |
| `GROUP BY` | Compute aggregations within groups | Stats by type |
| `JOIN` | Combine data from two tables | Attach English names to species IDs |

These same commands work on any SQL database — government microdata, financial records, or hospital datasets. The Pokédex is a fun playground, but the skills are the same ones economists use every day to query large, structured datasets.